<a href="https://colab.research.google.com/github/TomonoriGH/colab_git/blob/main/arbit_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. 設定とモデルの定義
入力層と出力層のサイズを指定し、シンプルなニューラルネットワークを定義します。

In [ ]:
!pip install torchinfo

In [ ]:
torch.set_default_device('cuda')

#### example

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# --- パラメータ設定 ---
INPUT_SIZE = 10   # 入力層の大きさ
OUTPUT_SIZE = 2   # 出力層の大きさ (例: 2クラス分類)
HIDDEN_SIZE = 64  # 隠れ層の大きさ
BATCH_SIZE = 16
LEARNING_RATE = 0.01
EPOCHS = 20

# モデルの定義
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("Model initialized.")

Model initialized.


#### selfmade

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchinfo import summary

# --- パラメータ設定 ---
INPUT_SIZE = 6   # 入力層の大きさ
OUTPUT_SIZE = 6   # 出力層の大きさ (例: 2クラス分類)
NUM_SAMPLE = 10000
HIDDEN_SIZE = 64  # 隠れ層の大きさ
BATCH_SIZE = 16
LEARNING_RATE = 0.01
EPOCHS = 20
LAYER_LENGTH = 5

model = nn.Sequential(*[
        nn.Linear(
              INPUT_SIZE if i == 0 else HIDDEN_SIZE,
              OUTPUT_SIZE if i == LAYER_LENGTH - 1 else HIDDEN_SIZE
        )
      if ii == 0 else
      nn.ReLU()
      for i in range(LAYER_LENGTH)
      for ii in range(1 if i == LAYER_LENGTH - 1 else 2)
    ])
# criterion = nn.CrossEntropyLoss()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("Model initialized.")
summary(model)

Model initialized.


Layer (type:depth-idx)                   Param #
Sequential                               --
├─Linear: 1-1                            448
├─ReLU: 1-2                              --
├─Linear: 1-3                            4,160
├─ReLU: 1-4                              --
├─Linear: 1-5                            4,160
├─ReLU: 1-6                              --
├─Linear: 1-7                            4,160
├─ReLU: 1-8                              --
├─Linear: 1-9                            390
Total params: 13,318
Trainable params: 13,318
Non-trainable params: 0

### 2. ミニバッチ生成関数とダミーデータ
学習に使用するサンプルデータと、バッチを取得する関数を用意します。

#### example

In [ ]:
# ダミーデータの作成 (入力: 100サンプル, 正解ラベル: 0 or 1)
X_train = torch.randn(100, INPUT_SIZE)
y_train = torch.randint(0, OUTPUT_SIZE, (100,))

def get_mini_batches(X, y, batch_size):
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    for i in range(0, len(X), batch_size):
        batch_idx = indices[i:i + batch_size]
        yield X[batch_idx], y[batch_idx]

print("Data and batch generator ready.")

#### selfmade

In [ ]:
def ylinside(l) :
  l = [str(e) for e in l]
  v1 = int("".join(l[:3])) * int("".join(l[3:]))
  return [int(e) for e in list(str(v1).zfill(6))]

xl,yl,X_train,y_train = [None] * 4

def gen_train_data() :
  global xl,yl,X_train,y_train
  xl = np.random.randint(0,10,size=(NUM_SAMPLE,INPUT_SIZE)).tolist()
  yl = [ylinside(xl[i]) for i in range(NUM_SAMPLE)]

  X_train = torch.tensor(xl,dtype=torch.float)
  y_train = torch.tensor(yl,dtype=torch.float)

def get_mini_batches(X, y, batch_size):
    gen_train_data()
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    for i in range(0, len(X), batch_size):
        batch_idx = indices[i:i + batch_size]
        yield X[batch_idx], y[batch_idx]


print("Data and batch generator ready.")

Data and batch generator ready.


### 3. 学習実行 (ボタン一つで実行相当)
このセルを実行することで学習が始まります。

In [ ]:
list(get_mini_batches(X_train,y_train,1))

In [ ]:
def train():
    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch_X, batch_y in get_mini_batches(X_train, y_train, BATCH_SIZE):
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss:.4f}")
    print("Training completed.")

train()

Epoch [5/20], Loss: 3664.2284
Epoch [10/20], Loss: 3646.5303
Epoch [15/20], Loss: 3647.8001
Epoch [20/20], Loss: 3679.6336
Training completed.


### 4. 使用・評価 (ボタン一つで実行相当)
学習済みモデルを使用して推論を行う例です。

#### example

In [ ]:
def evaluate():
    model.eval()
    # 新しい未知のデータを作成
    test_input = torch.randn(5, INPUT_SIZE)

    with torch.no_grad():
        predictions = model(test_input)
        # 最大値のインデックスを取得 (クラス予測)
        _, predicted_classes = torch.max(predictions, 1)

    print("Input Data Shape:", test_input.shape)
    print("Predicted Classes:", predicted_classes.numpy())

evaluate()

Input Data Shape: torch.Size([5, 10])
Predicted Classes: [1 0 1 0 1]


#### selfmade

In [ ]:
def evaluate():
    model.eval()
    # 新しい未知のデータを作成
    test_input,answer = next(get_mini_batches(X_train,y_train,1))
    print("input",test_input)
    print("answer",answer)

    with torch.no_grad():
        predictions = model(test_input)
        print(predictions)
        # 最大値のインデックスを取得 (クラス予測)
        _, predicted_classes = torch.max(predictions, 1)

    print("Input Data Shape:", test_input.shape)
    # print("Predicted Classes:", predicted_classes.numpy())

evaluate()

input tensor([[9., 5., 5., 6., 8., 7.]], device='cuda:0')
answer tensor([[6., 5., 6., 0., 8., 5.]], device='cuda:0')
tensor([[5.5493, 4.6150, 4.4131, 4.4162, 4.5720, 3.9202]], device='cuda:0')
Input Data Shape: torch.Size([1, 6])
